In [4]:
import os
from IPython.display import Markdown, HTML, display

from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.utilities import SQLDatabase
from langchain_openai import AzureChatOpenAI

In [7]:
print(os.getcwd())

/Users/manika.midha/my_practice/build_database_agent


In [10]:
from pathlib import Path
from sqlalchemy import create_engine
import pandas as pd

csv_path = Path("interact_csv_and_sql_data")/ "all-states-history.csv"
db_path = Path("interact_csv_and_sql_data")/ "db" / "test.db"

db_path.parent.mkdir(parents=True, exist_ok=True)
engine = create_engine(f'sqlite:///{db_path.resolve()}')

df = pd.read_csv(csv_path).fillna(value=0)
df.to_sql(
    "all-states-history",
    con=engine,
    if_exists="replace",
    index=False
)

20780

In [11]:
# prepare the SQL prompt
MSSQL_AGENT_PREFIX = """

You are an agent designed to interact with a SQL database.
## Instructions:
- Given an input question, create a syntactically correct {dialect} query
to run, then look at the results of the query and return the answer.
- Unless the user specifies a specific number of examples they wish to
obtain, **ALWAYS** limit your query to at most {top_k} results.
- You can order the results by a relevant column to return the most
interesting examples in the database.
- Never query for all the columns from a specific table, only ask for
the relevant columns given the question.
- You have access to tools for interacting with the database.
- You MUST double check your query before executing it.If you get an error
while executing a query,rewrite the query and try again.
- DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.)
to the database.
- DO NOT MAKE UP AN ANSWER OR USE PRIOR KNOWLEDGE, ONLY USE THE RESULTS
OF THE CALCULATIONS YOU HAVE DONE.
- Your response should be in Markdown. However, **when running  a SQL Query
in "Action Input", do not include the markdown backticks**.
Those are only for formatting the response, not for executing the command.
- ALWAYS, as part of your final answer, explain how you got to the answer
on a section that starts with: "Explanation:". Include the SQL query as
part of the explanation section.
- If the question does not seem related to the database, just return
"I don\'t know" as the answer.
- Only use the below tools. Only use the information returned by the
below tools to construct your query and final answer.
- Do not make up table names, only use the tables returned by any of the
tools below.

## Tools:

"""

In [12]:
MSSQL_AGENT_FORMAT_INSTRUCTIONS = """

## Use the following format:

Question: the input question you must answer.
Thought: you should always think about what to do.
Action: the action to take, should be one of [{tool_names}].
Action Input: the input to the action.
Observation: the result of the action.
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer.
Final Answer: the final answer to the original input question.

Example of Final Answer:
<=== Beginning of example

Action: query_sql_db
Action Input: 
SELECT TOP (10) [death]
FROM covidtracking 
WHERE state = 'TX' AND date LIKE '2020%'

Observation:
[(27437.0,), (27088.0,), (26762.0,), (26521.0,), (26472.0,), (26421.0,), (26408.0,)]
Thought:I now know the final answer
Final Answer: There were 27437 people who died of covid in Texas in 2020.

Explanation:
I queried the `covidtracking` table for the `death` column where the state
is 'TX' and the date starts with '2020'. The query returned a list of tuples
with the number of deaths for each day in 2020. To answer the question,
I took the sum of all the deaths in the list, which is 27437.
I used the following query

```sql
SELECT [death] FROM covidtracking WHERE state = 'TX' AND date LIKE '2020%'"
```
===> End of Example

"""

In [18]:
# call the Azure chat model and create the SQL agent

db = SQLDatabase.from_uri(f'sqlite:///{db_path.resolve()}')
llm = AzureChatOpenAI(
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    temperature=0, # reduces randomness, model becomes more consistent and less creative
    max_tokens=500, # limits the maximum length of the model’s response, controls cost and reduce rambling answers
    disabled_params={"stop": None}
)

agent_executor_sql = create_sql_agent(
    prefix = MSSQL_AGENT_PREFIX,
    format_instructions = MSSQL_AGENT_FORMAT_INSTRUCTIONS,
    llm = llm,
    db = db,
    agent_type="tool-calling",
    top_k = 10,
    verbose = True
)

In [16]:
QUESTION = """How may patients were hospitalized during October 2020
in New York, and nationwide as the total of all states?
Use the hospitalizedIncrease column
"""

In [19]:
# invoke the SQL model 
agent_executor_sql.invoke(QUESTION)



> Entering new SQL Agent Executor chain...



Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


all-states-history


Invoking: `sql_db_schema` with `{'table_names': 'all-states-history'}`



CREATE TABLE "all-states-history" (
	date TEXT, 
	state TEXT, 
	death FLOAT, 
	"deathConfirmed" FLOAT, 
	"deathIncrease" BIGINT, 
	"deathProbable" FLOAT, 
	hospitalized FLOAT, 
	"hospitalizedCumulative" FLOAT, 
	"hospitalizedCurrently" FLOAT, 
	"hospitalizedIncrease" BIGINT, 
	"inIcuCumulative" FLOAT, 
	"inIcuCurrently" FLOAT, 
	negative FLOAT, 
	"negativeIncrease" BIGINT, 
	"negativeTestsAntibody" FLOAT, 
	"negativeTestsPeopleAntibody" FLOAT, 
	"negativeTestsViral" FLOAT, 
	"onVentilatorCumulative" FLOAT, 
	"onVentilatorCurrently" FLOAT, 
	positive FLOAT, 
	"positiveCasesViral" FLOAT, 
	"positiveIncrease" BIGINT, 
	"positiveScore" BIGINT, 
	"positiveTestsAntibody" FLOAT, 
	"positiveTestsAntigen" FLOAT, 
	"positiveTestsPeopleAntibody" FLOAT, 
	"positiveTestsPeopleAntigen" FLOAT, 
	"positiveTestsViral" FLOAT, 
	recovered FLOAT, 
	"totalTestEncountersViral" FLOAT, 
	"totalTestEncountersViralIncrease" BIGINT, 
	"to


Invoking: `sql_db_query_checker` with `{'query': 'SELECT SUM(CASE WHEN state = \'NY\' THEN hospitalizedIncrease ELSE 0 END) AS new_york_hospitalized_oct_2020, SUM(hospitalizedIncrease) AS nationwide_hospitalized_oct_2020 FROM "all-states-history" WHERE date >= \'2020-10-01\' AND date <= \'2020-10-31\';'}`




SELECT SUM(CASE WHEN state = 'NY' THEN hospitalizedIncrease ELSE 0 END) AS new_york_hospitalized_oct_2020, SUM(hospitalizedIncrease) AS nationwide_hospitalized_oct_2020 FROM "all-states-history" WHERE date >= '2020-10-01' AND date <= '2020-10-31';


Invoking: `sql_db_query` with `{'query': 'SELECT SUM(CASE WHEN state = \'NY\' THEN hospitalizedIncrease ELSE 0 END) AS new_york_hospitalized_oct_2020, SUM(hospitalizedIncrease) AS nationwide_hospitalized_oct_2020 FROM "all-states-history" WHERE date >= \'2020-10-01\' AND date <= \'2020-10-31\';'}`


[(0, 53485)]

During **October 2020**:

- **New York:** **0** patients hospitalized
- **Nationwide total (all states):** **53,485** patients hospitalized

## Explanation:
I summed the `hospitalizedIncrease` values for all dates from **2020-10-01** through **2020-10-31**.  
For New York, I summed only rows where `state = 'NY'`.  
For the nationwide total, I summed `hospitalizedIncrease` across all states.

SQL query used:

```sql
SELECT
  SUM(CASE WHEN state = 'NY' THEN hospitalizedIncrease ELSE 0 END) AS new_york_hospitalized_oct_2020,
  SUM(hospitalizedIncrease) AS nationwide_hospitalized_oct_2020
FROM "all-states-history"
WHERE date >= '2020-10-01' AND date <= '2020-10-31';
```

> Finished chain.


{'input': 'How may patients were hospitalized during October 2020\nin New York, and nationwide as the total of all states?\nUse the hospitalizedIncrease column\n',
 'output': 'During **October 2020**:\n\n- **New York:** **0** patients hospitalized\n- **Nationwide total (all states):** **53,485** patients hospitalized\n\n## Explanation:\nI summed the `hospitalizedIncrease` values for all dates from **2020-10-01** through **2020-10-31**.  \nFor New York, I summed only rows where `state = \'NY\'`.  \nFor the nationwide total, I summed `hospitalizedIncrease` across all states.\n\nSQL query used:\n\n```sql\nSELECT\n  SUM(CASE WHEN state = \'NY\' THEN hospitalizedIncrease ELSE 0 END) AS new_york_hospitalized_oct_2020,\n  SUM(hospitalizedIncrease) AS nationwide_hospitalized_oct_2020\nFROM "all-states-history"\nWHERE date >= \'2020-10-01\' AND date <= \'2020-10-31\';\n```'}